# ML Challenge 

<img src="https://imageio.forbes.com/specials-images/imageserve/5ecd179f798e4c00060d2c7c/0x0.jpg?format=jpg&height=600&width=1200&fit=bounds" width="500" height="300">

In the bustling city of Financia, the Central Lending Institution (CLI) is the largest provider of loans to individuals and businesses. With a mission to support economic growth and financial stability, CLI processes thousands of loan applications every month. However, the traditional manual review process is time-consuming and prone to human error, leading to delays and inconsistencies in loan approvals.
To address these challenges, CLI has decided to leverage the power of machine learning to streamline their loan approval process. They have compiled a comprehensive dataset containing historical loan application records, including various factors such as credit scores, income levels, employment status, loan terms(measured in years), loan amounts, asset values, and the final loan status (approved or denied).


**Your task is to develop a predictive model that can accurately determine the likelihood of loan approval based on the provided features. By doing so, you will help CLI make faster, more accurate, and fairer lending decisions, ultimately contributing to the financial well-being of the community.**

It is recommended that you follow the typical machine learning workflow, though you are not required to strictly follow each steps: 
1. Data Collection: Gather the data you need for your model. (Already done for you)

2. Data Preprocessing: Clean and prepare the data for analysis. (Already done for you)

3. Exploratory Data Analysis (EDA): Understand the data and its patterns. (Partially done for you)

4. Feature Engineering: Create new features or modify existing ones to improve model performance. (Partially done for you)

5. Model Selection: Choose the appropriate machine learning algorithm.

6. Model Training: Train the model using the training dataset.

7. Model Evaluation: Evaluate the model's performance using a validation dataset.

8. Model Optimization: Optimize the model's parameters to improve performance.

9. Model Testing: Test the final model on a separate test dataset.

**Please include ALL your work and thought process in this notebook**

In [ ]:
# You may include any package you deem fit. We sugggest looking into Scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# sklearn will be imported later

## Dataset


In [ ]:
# DO NOT MODIFY
loan_data = pd.read_csv("../../data/loan_approval.csv")
loan_data

## EDA
Uncomment to see desired output. Add more analysis if you like

In [ ]:

import matplotlib.pyplot as plt

# ------ Display basic information ------
print(loan_data.columns)
print(loan_data.describe())

# ------ Check for missing values ------
print(loan_data.isnull().sum())

# ------ Visualize the distribution of loan status ------
loan_status_counts = loan_data['loan_status'].value_counts()
plt.bar(loan_status_counts.index, loan_status_counts.values)
plt.title('Distribution of Loan Status')
plt.xlabel('Loan Status')
plt.ylabel('Count')

# ------ Visualize the distribution of numerical features ------ 
loan_data.hist(bins=30, figsize=(20, 15))

# ------ Correlation matrix ------
corr_matrix = loan_data.corr(numeric_only=True)     # My fix
fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(corr_matrix, cmap='coolwarm')
fig.colorbar(cax)
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)

# ----- MORE (Encouraged but not required) ------
# TODO 

## Feature Engineering

You may want to convert categorical variables to numerical. For example, education takes on the value Graduate and Not Graduate. But we want it to be 0 or 1 for machine learning algorithms to use.

In [ ]:
loan_data['education'] = loan_data['education'].map({'Graduate': 1, 'Not Graduate': 0})
# Hint: Other categorical variables are self_employed and loan_status
# TODO

In [ ]:
loan_data['self_employed'].value_counts(), loan_data['loan_status'].value_counts()

In [ ]:
loan_data['self_employed'] = loan_data['self_employed'].map({'Yes': 1, 'No': 0})
loan_data['loan_status'] = loan_data['loan_status'].map({'Approved': 1, 'Rejected': 0})
loan_data

## Model Selection

You are free to use any classification machine learning models you like: Logistic Regression, Decision Trees/Random Forests, Support Vector Machines, KNN ... 

In [ ]:
# We are using SVMs
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# Data includes widely varying ranges (e.g. income_annum vs. loan_term) and should be normalized
classifier = make_pipeline(StandardScaler(),    # normalize data
                           SVC(kernel="linear") # to keep things simple
                            )     

## Model Training and Evaluation

In [ ]:
# We will use X for features, y for target
X_all = loan_data.copy()
X_all.drop('loan_status', inplace=True, axis=1)
print(X_all.shape)
X_all

In [ ]:
y_all = loan_data.copy()['loan_status']
print(y_all.shape)
y_all

In [ ]:
y_all.value_counts()

In [ ]:
# Create train/test splits (validation omitted for simplicity)
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42, shuffle=True)
X_train.shape, y_train.shape, X_test.shape, y_test.shape    # reminder: order convention is different from pytorch

In [ ]:
classifier.fit(X_train, y_train)

## Model Optimization and Testing

### Qualitative Measures

In [ ]:
y_pred_train = classifier.predict(X_train)
y_pred_train.shape, y_pred_train.sum()

In [ ]:
y_pred_test = classifier.predict(X_test)
y_pred_test.shape, y_pred_test.sum()

In [ ]:
# Display decision boundaries
# Adapted from https://scikit-learn.org/stable/auto_examples/svm/plot_iris_svc.html (by myself)
from sklearn.base import clone
from sklearn.inspection import DecisionBoundaryDisplay

def plot_decision_boundary(X: pd.DataFrame, y: pd.Series, feature_x: str, feature_y: str) -> None:
    """Plots a **projection** of the SVC's decision boundary w.r.t two given features"""
    # We need a separate model for visualization in 2D
    # Otherwise, we will get the following
    # `ValueError: n_features must be equal to 2. Got 12 instead.`
    X = X[[feature_x, feature_y]]
    viz_clf = clone(classifier)
    viz_clf.fit(X, y)

    disp = DecisionBoundaryDisplay.from_estimator(
        viz_clf,
        X,
        response_method="predict",
        multiclass_colors="coolwarm",
        alpha=0.8,
        xlabel=feature_x,
        ylabel=feature_y,
    )

    # Plot the support vectors.
    # For LinearSVC we compute the support vectors from the decision function, see
    # https://scikit-learn.org/dev/auto_examples/svm/plot_linearsvc_support_vectors.html
    support_vector_indices = viz_clf.named_steps["svc"].support_

    plt.scatter(
        X.iloc[support_vector_indices, 0],
        X.iloc[support_vector_indices, 1],
        c=y.iloc[support_vector_indices],
        cmap=plt.cm.coolwarm,
        edgecolors="k",
    )

    plt.title("SVC Decision Boundary (projection)")

In [ ]:
# A somewhat decent example
plot_decision_boundary(X_train, y_train, "cibil_score", "loan_amount")

In [ ]:
# However, variables are not always linearly separable
plot_decision_boundary(X_train, y_train, "residential_assets_value", "loan_amount")
# We may want to try other SVM kernels, or even other models

### Quantitative Measures

In [ ]:
# Raw binary classification metrics
from sklearn.metrics import *   # for convenience
results = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test),
    'f1': f1_score(y_test, y_pred_test),
}

print(f"Accuracy:   {results['accuracy']:.3f}")    # 0.918
print(f"Recall:     {results['recall']:.3f}")      # 0.924
print(f"F1 Score:   {results['f1']:.3f}")          # 0.934
# Surprisingly decent?!

In [ ]:
# Confusion matrix
conf_mat = confusion_matrix(y_test, y_pred_test)
conf_mat

In [ ]:
# Neat display
conf_disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_test,
    cmap="Blues"
)
plt.title("Confusion Matrix")
plt.show()